# 04 — End-to-End Cognitive Explorer with Marcus Aurelius (real embeddings)

The full EgoVault cognitive journey, applied to the complete *Meditations* of
Marcus Aurelius (`notebooks/assets/Marcus-Aurelius-Meditations.pdf`), using the
**real embedding provider** (Ollama `nomic-embed-text`, 768 dims) end to end — no
mock vectors anywhere in the pipeline:

1. **Episodic ingestion** (Tier 1 chunks, real embeddings)
2. **Automatic thematic segmentation** (`note_candidates` queue)
3. **Lease-based lock claiming** (TTL)
4. **Atomic conversion to an Obsidian Markdown note** (Tier 2)
5. **Human-in-the-loop validation** (`review_status`: unreviewed -> reviewed)
6. **Prefrontal recall** (`curate()`) with confidence weighting
7. **Verbatim evidence drill-down** (`get_chunks()`)
8. **Data-flow and confidence visualizations**
9. **Embedding-space projection** — why `curate()` escalated to chunks, visually

In [ ]:
import sys
import pypdf
import tempfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import load_settings
from infrastructure.context import build_context
from workflows.ingest import ingest
from tools.vault.list_note_candidates import list_note_candidates
from tools.vault.claim_note_candidate import claim_note_candidate
from tools.vault.create_note_from_candidate import create_note_from_candidate
from tools.vault.curate import curate
from tools.vault.get_chunks import get_chunks
from tools.vault.update_note import update_note
from core.schemas import NoteContentInput
from notebooks._lib.embedding_cache import wrap_ctx_embed, probe_provider

print("EgoVault End-to-End Explorer ready!")

## 1. Environment Isolation & PDF Ingestion (real embeddings)

In [ ]:
pdf_file = PROJECT_ROOT / "notebooks" / "assets" / "Marcus-Aurelius-Meditations.pdf"
reader = pypdf.PdfReader(str(pdf_file))
pages_text = [page.extract_text() for page in reader.pages]
full_pdf_text = "\n\n".join(filter(None, pages_text))

tmp_dir = tempfile.TemporaryDirectory()
tmp_path = Path(tmp_dir.name)

settings = load_settings()
vault_db = tmp_path / "vault.db"
sys_db = tmp_path / "system.db"
vault_dir = tmp_path / "vault"
media_dir = tmp_path / "media"
vault_dir.mkdir()
media_dir.mkdir()

from infrastructure.db import init_db, init_system_db
from infrastructure.vault_db import VaultDB
from infrastructure.vault_writer import write_note as _write_note

init_db(vault_db)
init_system_db(sys_db)

ctx = build_context(settings)
ctx.db = VaultDB(vault_db)
ctx.system_db_path = sys_db
ctx.vault_path = vault_dir
ctx.media_path = media_dir
ctx.write_note = _write_note

probe_provider(ctx)
wrap_ctx_embed(ctx)  # real Ollama embeddings, disk-cached — no mock fallback

source = ingest("texte", full_pdf_text, ctx, title="Meditations of Marcus Aurelius")
print(f"Source Ingested: {source.title} (Status: {source.status})")

## 2. Note Candidate Queue & Lease-Based Locking

In [ ]:
cands = list_note_candidates(ctx, source_uid=source.uid)
total_chunks = sum(len(c.chunk_uids) for c in cands)
print(f"Chunks: {total_chunks} -- Candidates generated: {len(cands)}\n")
for c in cands[:6]:
    print(f" - #{c.sequence_index + 1}: '{c.label[:55]}...' ({len(c.chunk_uids)} chunks)")

cand_to_convert = cands[0]
claimed = claim_note_candidate(cand_to_convert.uid, ctx, session_id="human_explorer", is_human=True)
print(f"\nCandidate #1 locked by: {claimed.claimed_by} (Status: {claimed.status})")

## 3. Atomic Conversion & `review_status` Validation

In [ ]:
first_chunk_text = next(c.content for c in ctx.db.get_chunks(cand_to_convert.chunk_uids))
content = NoteContentInput(
    title="Book I: Debt of Gratitude and Mentors",
    docstring="Reflections of Marcus Aurelius on the virtues inherited from his ancestors and Stoic teachers.",
    body=first_chunk_text[:1200],
    tags=["stoicism", "gratitude", "marcus-aurelius"],
)
note_res = create_note_from_candidate(
    candidate_uid=cand_to_convert.uid,
    content=content,
    ctx=ctx,
    session_id="human_explorer",
    note_type="synthese",
    tags=["stoicism", "gratitude", "marcus-aurelius"],
)
note = note_res.note
print(f"Note created: {note.title} (UID: {note.uid[:8]}...)")
print(f"Initial review status: {note.review_status}")

update_note(note.uid, {"review_status": "reviewed"}, ctx)
reviewed_note = ctx.db.get_note(note.uid)
print(f"Updated review status: {reviewed_note.review_status}")

## 4. Prefrontal Recall `curate()` & Evidence Drill-Down `get_chunks()`

With real embeddings, watch the confidence score — it should read meaningfully
higher than the ~0.19 the previous bag-of-words mock produced, because real
semantic similarity actually separates relevant from irrelevant text.

In [ ]:
curated = curate("duty virtue and death", ctx, limit=5)
print(f"Query: '{curated.query}'")
print(f"Confidence: {curated.confidence} (reviewed note weight=1.0 vs unreviewed=0.7)")
print(f"\nSources retrieved ({len(curated.sources)}):")
for s in curated.sources:
    print(f" - [{s.tier}] {s.title[:45]}... (distance: {s.distance:.4f})")

proof_chunks = get_chunks(cand_to_convert.chunk_uids[:3], ctx)
print(f"\nVerbatim proof drill-down via get_chunks() ({len(proof_chunks)} chunks):")
for ch in proof_chunks:
    print(f" - Chunk pos={ch.position} (UID: {ch.uid[:8]}...): '{ch.content[:80]}...'")

## 5. Pipeline & Confidence Visualizations (real numbers)

The chunk count in this chart comes from the real segmented candidate queue
(`sum(len(c.chunk_uids) for c in cands)`), not a fabricated multiplication —
it must equal the "Chunks" count printed in section 2.

In [ ]:
%matplotlib inline
assets_dir = PROJECT_ROOT / "notebooks" / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 4.5), dpi=120)
stages = ['PDF (128 pages)', 'SOTA Chunks', 'Note Candidates', 'Active Notes']
page_count = len(reader.pages)
counts = [page_count, total_chunks, len(cands), 1]
colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b8']

bars = ax.bar(stages, counts, color=colors, width=0.55)
ax.set_title("EgoVault Cognitive Architecture Data Flow", fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel("Entity Count", fontsize=11)
ax.grid(True, linestyle='--', alpha=0.3, axis='y')

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.5, f"{int(yval)}", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.show()

In [ ]:
fig2, ax2 = plt.subplots(figsize=(10, 4), dpi=120)
if curated.sources:
    src_titles = [f"[{s.tier[:4]}] {s.title[:25]}..." for s in curated.sources]
    similarities = [max(0.0, 1.0 - s.distance) for s in curated.sources]

    y_pos = np.arange(len(src_titles))
    ax2.barh(y_pos, similarities, color='#2ca02c', alpha=0.8, height=0.55)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(src_titles, fontsize=9)
    ax2.invert_yaxis()
    ax2.set_xlabel("Similarity Score ($1.0 - \\text{distance}$)", fontsize=11)
    ax2.set_title(f"curate() Retrieval Results (Confidence: {curated.confidence})", fontsize=13, fontweight='bold', pad=12)
    ax2.grid(True, linestyle='--', alpha=0.3, axis='x')

    for i, sim in enumerate(similarities):
        ax2.text(sim + 0.01, i, f"{sim:.3f}", va='center', fontsize=9, fontweight='bold')

plt.show()

## 6. Embedding-Space Projection — Why curate() Escalated (or Didn't)

A PCA projection of the query vector alongside every retrieved source's real
embedding. This makes the escalation decision visible instead of implicit:
if fewer than `escalation_min_notes` notes land close enough to the query
(`escalation_max_distance`), `curate()` pulls in raw chunks too.

In [ ]:
proj_texts = [curated.query] + [s.content[:500] for s in curated.sources]
proj_labels = ["QUERY"] + [f"[{s.tier}] {s.title[:20]}" for s in curated.sources]
proj_tiers = ["query"] + [s.tier for s in curated.sources]
proj_vectors = [ctx.embed(t) for t in proj_texts]

coords_proj = PCA(n_components=2).fit_transform(proj_vectors)
color_map = {"query": "#c44e52", "note": "#55a868", "chunk": "#4c72b0"}

fig3, ax3 = plt.subplots(figsize=(8, 7), dpi=120)
for i, (x, y) in enumerate(coords_proj):
    tier = proj_tiers[i]
    ax3.scatter(x, y, s=180 if tier == "query" else 100, color=color_map[tier],
                edgecolors='black', linewidth=0.8, alpha=0.85,
                marker='*' if tier == "query" else 'o')
    ax3.annotate(proj_labels[i], (x, y), fontsize=8, xytext=(6, 4), textcoords='offset points')

n_note_sources = sum(1 for s in curated.sources if s.tier == "note")
escalated = n_note_sources < ctx.settings.system.curate.escalation_min_notes
ax3.set_title(
    f"Query vs. Retrieved Sources (PCA of real embeddings)\n"
    f"{n_note_sources} relevant note(s) < escalation_min_notes="
    f"{ctx.settings.system.curate.escalation_min_notes} -> escalated to chunks: {escalated}",
    fontsize=11, fontweight='bold', pad=12,
)
legend_handles = [
    plt.Line2D([0], [0], marker='*', color='w', markerfacecolor=color_map['query'], markersize=14, label='Query'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map['note'], markersize=10, label='Note (tier 2)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map['chunk'], markersize=10, label='Chunk (tier 1)'),
]
ax3.legend(handles=legend_handles, loc='best', fontsize=9)
ax3.grid(True, linestyle='--', alpha=0.3)
plt.show()

tmp_dir.cleanup()

## 7. Takeaway

Every number in this notebook comes from the real pipeline: real embeddings, real
segmentation thresholds, real confidence weighting. The previous version used a
hash-based bag-of-words mock and a fabricated chunk count — both silently
overstated how separable and how large the corpus looked. This is what the
engine actually does.